In [39]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import json
import pickle

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

PyTorch: 2.11.0+cu130
CUDA disponible: True


In [40]:
BASE_DATA_DIR = '../data'
OUTPUT_DIR    = '../data/processed'
MODELS_DIR    = '../models'
RESULTS_DIR   = '../results'

HOURS_BEFORE  = 48 # Horas previas al ingreso en UCI
MIN_HOURS     = 6 # Estancias con al menos N horas de datos
N_CONTROLS    = 29_966  # Controles (ratio 1:1 con sepsis)

# Hiperparámetros
BATCH_SIZE    = 64
EPOCHS        = 100
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
# EL LSTM mantiene en cada timestep un vector de estado oculto, con este valor definimos la dimension de la "memoria de trabajo"
# En este caso la capa LSTM recibe 32 features por timestep, y produce un vector de 128 valores en cada paso temporal
HIDDEN_SIZE   = 32
NUM_LAYERS    = 1
DROPOUT       = 0.3
PATIENCE = 7

RANDOM_STATE_SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

Dispositivo: cuda


In [41]:
features_all = pd.read_parquet(f'{OUTPUT_DIR}/features_all.parquet')
cohort_all   = pd.read_parquet(f'{OUTPUT_DIR}/cohort_all.parquet')

# Temperature Celsius: 99.8% nulos en NB1 → eliminada
FEATURE_COLS = [
    c for c in features_all.columns
    if c not in ['stay_id', 'hour_bucket', 'Temperature Celsius']
]
N_FEATURES = len(FEATURE_COLS)

print(f'Estancias totales: {cohort_all["stay_id"].nunique():,}')
print(f'  Sepsis  (label=1): {(cohort_all["label"]==1).sum():,}')
print(f'  Control (label=0): {(cohort_all["label"]==0).sum():,}')
print(f'Features ({N_FEATURES}): {FEATURE_COLS}')

Estancias totales: 59,932
  Sepsis  (label=1): 29,966
  Control (label=0): 29,966
Features (16): ['Arterial Blood Pressure mean', 'GCS - Motor Response', 'GCS - Verbal Response', 'Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Bilirubin, Total', 'Creatinine', 'Lactate', 'Platelet Count', 'Urea Nitrogen', 'pH', 'pO2']


In [42]:
# Agrupamos los datos por estancia y contamos las secuencias horarias que tenemos (cuantas horas tiene cada estancia)
stays_coverage = features_all.groupby('stay_id')['hour_bucket'].count()
# Filtramos por el minimo de horas establecido previamente y obtenemos una lista de identificadores
valid_stay_ids = stays_coverage[stays_coverage >= MIN_HOURS].index

# Extraemos etiquetas y pacientes a partir de las estancias filtradas
stay_labels_v = cohort_all[cohort_all['stay_id'].isin(valid_stay_ids)][
    ['stay_id', 'label', 'subject_id']
].reset_index(drop=True)

# Obtenemos listas para poder usar GroupShuffleSplit
all_stays = stay_labels_v['stay_id'].values # Lista de identificadores de estancias
all_labels = stay_labels_v['label'].values # Sepsis o control (sepsis / no sepsis) [0, 1, 1, 0...]
all_subjects = stay_labels_v['subject_id'].values # Lista de identificadores de pacientes

# Split 1: train + val (80%) / test (20%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE_SEED)
tv_idx, test_idx = next(gss1.split(all_stays, all_labels, groups=all_subjects))

# Split 2: train (75%) / val (25%) del 80% anterior → 60/20/20 global
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE_SEED)
tr_idx, val_idx = next(gss2.split(
    all_stays[tv_idx], all_labels[tv_idx], groups=all_subjects[tv_idx]
))

# Extraemos los identificadores de estancia
train_stays = all_stays[tv_idx][tr_idx] # 80% → 75% de ese 80% = 60% global
val_stays = all_stays[tv_idx][val_idx] # 80% → 25% de ese 80% = 20% global
test_stays = all_stays[test_idx] # 20% global

# Creamos un mapa de estancia → paciente y se usa para obtener que pacientes hay en cada conjunto.
s2sub = cohort_all.set_index('stay_id')['subject_id']
tr_sub = set(s2sub[s2sub.index.isin(train_stays)]) # pacientes en train
va_sub = set(s2sub[s2sub.index.isin(val_stays)]) # pacientes en val
te_sub = set(s2sub[s2sub.index.isin(test_stays)]) # pacientes en test

print(f'Train: {len(train_stays):,} estancias | {len(tr_sub):,} pacientes')
print(f'Val: {len(val_stays):,} estancias | {len(va_sub):,} pacientes')
print(f'Test: {len(test_stays):,} estancias | {len(te_sub):,} pacientes')

Train: 2,193 estancias | 2,004 pacientes
Val: 732 estancias | 668 pacientes
Test: 736 estancias | 669 pacientes


In [43]:
def build_tensor(features_df, stay_ids, feature_cols, hours=HOURS_BEFORE):
    """Convierte features_df en tensor numpy (n, hours, n_features)."""

    stay_ids = list(stay_ids) # Lista de identificadores de estancias

    n = len(stay_ids) # Numero de estancias
    F = len(feature_cols) # Numero de variables
    X = np.full((n, hours, F), np.nan, dtype=np.float32) # Matriz rellena con NaN con forma (estancias, 48 horas, N variables)
    
    # Asignamos a cada estancia su posicion en la matriz
    id_to_idx = pd.Series(np.arange(n), index=stay_ids) # Secuencia de posiciones, establecemos como indice el identificador de estancia
    col_to_idx = {c: j for j, c in enumerate(feature_cols)} # Crea una secuencia de posiciones para las variables

    # Filtramos solo las filas relevantes (solo las secuencias dentro de la ventana de 48h)
    df = features_df[
        features_df['stay_id'].isin(set(stay_ids)) &
        features_df['hour_bucket'].between(-hours, -1)
    ].copy()
    
    df['_i'] = id_to_idx[df['stay_id'].values].values # Añadir la posición de cada estancia en la matriz
    
    # Convertir hour_bucket de negativo a positivo: -48 → 0, -1 → 47
    # para que sirva como índice directo en el eje temporal del cubo
    df['_t'] = (df['hour_bucket'] + hours).astype(int)

    df = df.dropna(subset=['_i']) # Eliminamos filas cuyo stay_id no tiene posición asignada (no deberían existir)

    # Rellenar la matriz variable a variable
    for col, j in col_to_idx.items():
        if col not in df.columns: # Saltar si la variable no está en los datos
            continue
        
        # Extraemos solo las filas sin valores nulos
        sub = df[['_i', '_t', col]].dropna(subset=[col])

        i_idx = sub['_i'].astype(int).values # Índice de estancia
        t_idx = sub['_t'].astype(int).values # Índice de hora
        vals = sub[col].values.astype(np.float32) # Valor clínico
        
        # Filtro de seguridad: descartar índices fuera del rango válido [0, 47]
        mask = (t_idx >= 0) & (t_idx < hours)

        # Colocar cada valor en su celda exacta de la matriz [estancia, hora, variable]
        X[i_idx[mask], t_idx[mask], j] = vals[mask]

    return X

# Construir un tensor para cada conjunto (entrenamiento, validacion y test)
X_train_raw = build_tensor(features_all, train_stays, FEATURE_COLS)
X_val_raw = build_tensor(features_all, val_stays, FEATURE_COLS)
X_test_raw = build_tensor(features_all, test_stays, FEATURE_COLS)

# Extraemos las etiquetas (0/1) para cada conjunto usando el stay_id como identificador
label_map = stay_labels_v.set_index('stay_id')['label']
y_train = label_map[train_stays].values.astype(np.float32)
y_val = label_map[val_stays].values.astype(np.float32)
y_test = label_map[test_stays].values.astype(np.float32)

print(f'X_train: {X_train_raw.shape} | positivos: {y_train.mean():.2%}')
print(f'X_val: {X_val_raw.shape} | positivos: {y_val.mean():.2%}')
print(f'X_test: {X_test_raw.shape} | positivos: {y_test.mean():.2%}')

X_train: (2193, 48, 16) | positivos: 59.74%
X_val: (732, 48, 16) | positivos: 62.70%
X_test: (736, 48, 16) | positivos: 58.42%


In [ ]:
def apply_forward_fill_vectorized(X):
    """
    X : (n, T, F) con NaN.
    Convierte cada secuencia (T, F) a DataFrame, aplica ffill y reconstituye.
    """
    n, T, F = X.shape
    X_ff = X.copy()

    for i in range(n):
        df_tmp = pd.DataFrame(X[i])
        df_ff  = df_tmp.ffill(axis=0)
        X_ff[i] = df_ff.values

    return X_ff

# Calcular máscaras ANTES del forward fill (desde los NaN originales)
# La máscara debe reflejar mediciones reales, no valores propagados
M_train = (~np.isnan(X_train_raw)).astype(np.float32)
M_val   = (~np.isnan(X_val_raw)).astype(np.float32)
M_test  = (~np.isnan(X_test_raw)).astype(np.float32)

X_train_ff = apply_forward_fill_vectorized(X_train_raw)
X_val_ff   = apply_forward_fill_vectorized(X_val_raw)
X_test_ff  = apply_forward_fill_vectorized(X_test_raw)

pct_nan_antes   = np.isnan(X_train_raw).mean()
pct_nan_despues = np.isnan(X_train_ff).mean()
print(f'NaN antes del ffill:   {pct_nan_antes:.1%}')
print(f'NaN después del ffill: {pct_nan_despues:.1%}')

In [ ]:
def apply_missingness_mask(X, train_medians=None, external_mask=None):
    """
    Estrategia de imputación con missingness mask:
    1. Construye máscara binaria M: 1 = valor real, 0 = imputado.
    2. Rellena NaN con la mediana global del training (valor neutro).
    3. Concatena [X_imputed | M] → shape (n, T, 2*F).
    La máscara preserva la señal de ausencia como feature explícito.

    external_mask: máscara pre-computada desde el tensor original con NaN.
    Pasar siempre que X haya sido modificado antes (e.g. forward fill),
    para que la máscara refleje mediciones reales y no valores propagados.
    """

    _, _, F = X.shape

    # Usar máscara externa si se proporciona; si no, calcularla desde X
    M = external_mask if external_mask is not None else (~np.isnan(X)).astype(np.float32)

    X_imp = X.copy()

    if train_medians is None:
        flat = X_imp.reshape(-1, F)
        train_medians = np.nanmedian(flat, axis=0)
        train_medians = np.where(np.isnan(train_medians), 0.0, train_medians)

    for j in range(F):
        mask_nan = np.isnan(X_imp[:, :, j])
        X_imp[:, :, j][mask_nan] = train_medians[j]

    X_out = np.concatenate([X_imp, M], axis=2).astype(np.float32)
    return X_out, train_medians

# Pasar la máscara original (calculada antes del ffill) para no corromper la señal de ausencia
X_train_masked, train_medians = apply_missingness_mask(X_train_ff, external_mask=M_train)
X_val_masked,  _ = apply_missingness_mask(X_val_ff,  train_medians, external_mask=M_val)
X_test_masked, _ = apply_missingness_mask(X_test_ff, train_medians, external_mask=M_test)

F = len(FEATURE_COLS)
print(f'Shape tras aplicar mask: {X_train_masked.shape} → input_size al LSTM: {X_train_masked.shape[2]}')

In [46]:
# Solo se normalizan los 16 valores clínicos — las 16 máscaras ya son binarias {0,1} y no deben normalizarse
n_tr, T, F2 = X_train_masked.shape
F = F2 // 2  # Los primeros 16 canales son valores, los últimos 16 son máscaras

scaler = StandardScaler()  # Normalización z-score: media 0, desviación estándar 1

def scale_masked(X, scaler, F, fit=False):
    n, T, _ = X.shape
    X_vals = X[:, :, :F].reshape(-1, F)  # Extraer las 16 variables
    X_mask = X[:, :, F:] # Separar las 16 máscaras

    if fit:
        # En train: calcular media y desviación estándar y aplicar la normalización
        X_vals_scaled = scaler.fit_transform(X_vals)
    else:
        # En val y test: aplicar la normalización con los parámetros calculados en train
        # (evita data leakage: val/test no influyen en el cálculo de la normalización)
        X_vals_scaled = scaler.transform(X_vals)

    X_vals_scaled = X_vals_scaled.reshape(n, T, F).astype(np.float32)  # Restaurar forma 3D

    # Volver a concatenar valores normalizados con las máscaras sin tocar
    return np.concatenate([X_vals_scaled, X_mask], axis=2).astype(np.float32)

# Aplicar normalización: fit solo en train, transform en val y test
X_train_norm = scale_masked(X_train_masked, scaler, F, fit=True)
X_val_norm   = scale_masked(X_val_masked,   scaler, F, fit=False)
X_test_norm  = scale_masked(X_test_masked,  scaler, F, fit=False)

N_INPUT = X_train_norm.shape[2]  # 32 = 16 valores normalizados + 16 máscaras binarias
print(f'Input con mascaras binarias al LSTM: {N_INPUT}  (shape: {X_train_norm.shape})')
print(f'Máscaras — rango: [{X_train_norm[:,:,F:].min():.0f}, {X_train_norm[:,:,F:].max():.0f}]')

Input con mascaras binarias al LSTM: 32  (shape: (2193, 48, 32))
Máscaras — rango: [0, 1]


In [ ]:
def add_delta_features(x_imputed, mask):
    """
    x_imputed : (n, T, F) - valores imputados y normalizados
    mask      : (n, T, F) - máscara binaria (1 = medición real, 0 = imputado)
    Devuelve  : (n, T, F) - deltas calculadas solo entre mediciones reales
    """
    n, T, F = x_imputed.shape
    x_delta = np.zeros_like(x_imputed)

    for i in range(n):
        for f in range(F):
            last_real_val = None
            for t in range(T):
                if mask[i, t, f] == 1:
                    if last_real_val is not None:
                        x_delta[i, t, f] = x_imputed[i, t, f] - last_real_val
                    last_real_val = x_imputed[i, t, f]

    return x_delta

X_delta_train = add_delta_features(X_train_norm[:, :, :F], X_train_norm[:, :, F:])
X_delta_val   = add_delta_features(X_val_norm[:, :, :F],   X_val_norm[:, :, F:])
X_delta_test  = add_delta_features(X_test_norm[:, :, :F],  X_test_norm[:, :, F:])

# Concatenar: [valores (16) | máscaras (16) | deltas (16)] → (n, 48, 48)
X_train_final = np.concatenate([X_train_norm, X_delta_train], axis=2)
X_val_final   = np.concatenate([X_val_norm,   X_delta_val],   axis=2)
X_test_final  = np.concatenate([X_test_norm,  X_delta_test],  axis=2)

print(f'Shape con deltas: {X_train_final.shape} → input_size al LSTM: {X_train_final.shape[2]}')

In [ ]:
class SepsisDataset(Dataset):
    """Clase que alimenta al modelo durante el entrenamiento"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# X_train_norm  → forward fill + mask (32 canales)   ← experimento actual
# X_train_final → forward fill + mask + deltas (48 canales)
X_IN_TRAIN, X_IN_VAL, X_IN_TEST = X_train_norm, X_val_norm, X_test_norm

train_ds = SepsisDataset(X_IN_TRAIN, y_train)
val_ds   = SepsisDataset(X_IN_VAL,   y_val)
test_ds  = SepsisDataset(X_IN_TEST,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos

N_INPUT = X_IN_TRAIN.shape[2]
print(f'Input shape: {X_IN_TRAIN.shape} → N_INPUT: {N_INPUT}')
print(f'Train batches: {len(train_loader)} × {BATCH_SIZE}')

In [52]:
class LSTMImproved(nn.Module):

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: int = DROPOUT
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )

        combined_size = hidden_size * 4

        self.dropout = nn.Dropout(dropout)

        self.bn1 = nn.BatchNorm1d(combined_size)

        self.fc1 = nn.Linear(combined_size, 64)

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        out, _ = self.lstm(x)

        avg_pool = torch.mean(out, dim=1) # Promedio de las 48 horas
        max_pool, _ = torch.max(out, dim=1) # El valor máximo/pico de las 48 horas

        # Concatenamos ambos para la capa lineal
        combined = torch.cat([avg_pool, max_pool], dim=1) # shape: (batch, hidden_size * 4)

        out = self.dropout(combined)

        out = self.bn1(combined)

        out = self.relu(self.fc1(out))

        return self.fc2(out).squeeze(-1)
    
model = LSTMImproved(input_size=N_INPUT).to(DEVICE)
print(model)

LSTMImproved(
  (lstm): LSTM(16, 32, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [53]:
def train_model(model, model_name, pos_weight_value):
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    # Función de pérdida: Binary Cross-Entropy con logits
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Optimizador Adam: ajusta los pesos del modelo en cada paso usando el gradiente
    # lr=tasa de aprendizaje, weight_decay=regularización L2 para evitar overfitting
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Scheduler: reduce la tasa de aprendizaje a la mitad si la pérdida de validación
    # no mejora durante 3 épocas consecutivas — permite afinar el modelo en etapas tardías
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    # Historial de métricas por época para graficar las curvas de entrenamiento
    history          = {'train_loss': [], 'val_loss': [], 'val_auroc': []}
    best_val_loss    = float('inf')  # Mejor pérdida de validación vista hasta ahora
    patience_counter = 0             # Contador de épocas sin mejora para early stopping
    best_path        = f'{MODELS_DIR}/{model_name}.pt'  # Ruta donde guardar el mejor modelo

    # Entrenamiento por épocas, se define el numero de épocas previamente (50)
    for epoch in range(1, EPOCHS + 1):

        # Fase de entrenamiento
        model.train()  # Activar modo entrenamiento (dropout activo)
        train_loss    = 0.0
        train_samples = 0

        """
        Toma los features y objetivo de los datos de entrenamiento
        Por cada fila se aplica todo el proceso de:
        Limpiar gradientes -> Calcular pérdida -> Calcular gradientes -> Recortar gradientes -> Actualizar pesos del modelo -> Acumular pérdida ponderada -> Acumular número de muestras procesadas
        """
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)  # Mover batch a GPU/CPU
            optimizer.zero_grad() # Limpiamos gradientes del paso anterior
            loss = criterion(model(X_b), y_b) # Calculamos pérdida del batch
            loss.backward() # Calculamos gradientes (backpropagation)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Recortar gradientes para estabilidad
            optimizer.step() # Actualizamos pesos del modelo
            train_loss    += loss.item() * len(y_b) # Acumular pérdida ponderada por tamaño de batch
            train_samples += len(y_b) # Acumular número de muestras procesadas

        train_loss /= train_samples # Pérdida media real

        # Fase de validación
        model.eval()  # Activar modo evaluación (dropout desactivado)
        val_loss  = 0.0
        val_probs, val_true = [], []

        with torch.no_grad():  # Desactivar cálculo de gradientes para ahorrar memoria
            """
            1. Toma los features y objetivo de los datos de validación
            2. Convierte logits a probabilidades [0, 1] y las acumula
            3. Acumula las etiquetas reales (1 = sepsis, 0 = control)
            """
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                logits = model(X_b)
                val_loss += criterion(logits, y_b).item() * len(y_b)
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())  # Convertir logits a probabilidades [0,1]
                val_true.extend(y_b.cpu().numpy())

        # Pérdida media por estancia sobre todo el conjunto de validación
        val_loss /= len(val_loader.dataset)
        # Calculamos AUROC de validación solo si hay ambas clases presentes
        val_auroc = roc_auc_score(val_true, val_probs) if len(set(val_true)) > 1 else 0.0

        # Guardamos métricas de la época en el historial
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auroc'].append(val_auroc)

        # --- GESTIÓN DE SCHEDULER ---
        # Guardamos el LR antes del step para ver si cambia
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']

        if new_lr < old_lr:
            print(f'--- Tasa de aprendizaje reducida a: {new_lr:.2e} ---')

        # --- GESTIÓN DE EARLY STOPPING ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
            # print(f"Mejora detectada. Modelo guardado.") # Opcional
        else:
            patience_counter += 1

        # Print de progreso
        if epoch % 1 == 0: # Imprimimos cada época para ver el comportamiento del scheduler
            print(f'Epoch {epoch:3d}/{EPOCHS} | '
                  f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
                  f'Val AUROC: {val_auroc:.4f} | LR: {new_lr:.2e} | P: {patience_counter}')

        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch}. El modelo no mejoró en {PATIENCE} épocas.')
            break

    print(f'\nMejor val_loss: {best_val_loss:.4f}')
    return {
        'history': history,
        'path': best_path,
        'model': model,
        'pos_weight': pos_weight.item()
    }

models = {}

# n_neg / max(n_pos, 1)

model_name = 'best_lstm_bidirectional'
models[model_name] = train_model(model, model_name, n_neg / max(n_pos, 1))

Epoch   1/100 | Train Loss: 0.5478 | Val Loss: 0.5406 | Val AUROC: 0.6092 | LR: 5.00e-04 | P: 0
Epoch   2/100 | Train Loss: 0.5294 | Val Loss: 0.5252 | Val AUROC: 0.6262 | LR: 5.00e-04 | P: 0
Epoch   3/100 | Train Loss: 0.5167 | Val Loss: 0.5147 | Val AUROC: 0.6370 | LR: 5.00e-04 | P: 0
Epoch   4/100 | Train Loss: 0.5103 | Val Loss: 0.5152 | Val AUROC: 0.6506 | LR: 5.00e-04 | P: 1
Epoch   5/100 | Train Loss: 0.4997 | Val Loss: 0.5074 | Val AUROC: 0.6549 | LR: 5.00e-04 | P: 0
Epoch   6/100 | Train Loss: 0.4918 | Val Loss: 0.5052 | Val AUROC: 0.6568 | LR: 5.00e-04 | P: 0
Epoch   7/100 | Train Loss: 0.4838 | Val Loss: 0.4987 | Val AUROC: 0.6567 | LR: 5.00e-04 | P: 0
Epoch   8/100 | Train Loss: 0.4784 | Val Loss: 0.5175 | Val AUROC: 0.6514 | LR: 5.00e-04 | P: 1
Epoch   9/100 | Train Loss: 0.4725 | Val Loss: 0.5079 | Val AUROC: 0.6493 | LR: 5.00e-04 | P: 2
Epoch  10/100 | Train Loss: 0.4657 | Val Loss: 0.5066 | Val AUROC: 0.6514 | LR: 5.00e-04 | P: 3
--- Tasa de aprendizaje reducida a: 2.50

In [54]:
def show_model_metrics_optimized(model, model_name, model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    test_probs, test_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            logits = model(X_b.to(DEVICE))
            test_probs.extend(torch.sigmoid(logits).cpu().numpy())
            test_true.extend(y_b.numpy())

    test_probs = np.array(test_probs)
    test_true = np.array(test_true)

    # Cálculo automático del umbral óptimo (Youden's J statistic)
    fpr, tpr, thresholds = roc_curve(test_true, test_probs)
    best_thresh = thresholds[np.argmax(tpr - fpr)]

    preds = (test_probs >= best_thresh).astype(int)

    print(f'=== LSTM {model_name} (Umbral Opt: {best_thresh:.3f}) ===')
    print(f'AUROC: {roc_auc_score(test_true, test_probs):.4f}')
    print(classification_report(test_true, preds, target_names=['Control', 'Sepsis']))

for model_name, values in models.items():
    show_model_metrics_optimized(
        model=values['model'],
        model_name=model_name,
        model_path=values['path'],
    )

=== LSTM best_lstm_bidirectional (Umbral Opt: 0.467) ===
AUROC: 0.7023
              precision    recall  f1-score   support

     Control       0.71      0.43      0.54       306
      Sepsis       0.68      0.88      0.77       430

    accuracy                           0.69       736
   macro avg       0.70      0.65      0.65       736
weighted avg       0.70      0.69      0.67       736

